PRIMER PARCIAL: REGRESIÓN

In [40]:
# PyTorch para construir y entrenar redes neuronales
import torch

# Para crear Dataset y DataLoader
from torch.utils.data import Dataset, DataLoader

# Librerías para manipulación de datos
import numpy as np
import pandas as pd

# Librería para gráficos
from matplotlib import pyplot as plt

# Barra de progreso
from tqdm import tqdm
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo utilizado:", device)

Dispositivo utilizado: cpu


In [41]:
data  = pd.read_csv('DEMO_L.csv')
X     = data.iloc[:, :-1].values.astype(np.float32)
y_raw = data.iloc[:, -1].values
#Binarización: 1 = mitad superior (> 50), 0 = mitad inferior
y = (y_raw < 1.0).astype(int)

print(f"X: {X.shape}  |  y: {y.shape}")
print(f"Clase 0: {(y==0).sum()}  |  Clase 1: {(y==1).sum()}")


X: (11933, 27)  |  y: (11933,)
Clase 0: 9995  |  Clase 1: 1938


In [42]:
# Normalización y split 75/25
mu, sigma = X.mean(axis=0), X.std(axis=0)
X_norm    = (X - mu) / sigma

np.random.seed(42)
idx     = np.random.permutation(len(X_norm))
m_train = int(0.75* len(X_norm))
X_train, X_test = X_norm[idx[:m_train]], X_norm[idx[m_train:]]
y_train, y_test = y[idx[:m_train]],      y[idx[m_train:]]

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

Train: (8949, 27)  |  Test: (2984, 27)


C:\Users\usuario\AppData\Local\Temp\ipykernel_85332\1188786563.py:3: RuntimeWarning: invalid value encountered in divide
  X_norm    = (X - mu) / sigma


In [43]:
#Exploración inicial
print("\nColumnas disponibles:")
print(list(data.columns))

print("\nValores nulos por columna:")
print(data.isnull().sum())

print("\nEstadísticas descriptivas de variables clave:")

display(
    data[['RIDAGEYR','DMDHHSIZ','INDFMPIR']].describe()
)


Columnas disponibles:
['Unnamed: 0', 'SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN', 'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'RIDEXAGM', 'DMQMILIZ', 'DMDBORN4', 'DMDYRUSR', 'DMDEDUC2', 'DMDMARTZ', 'RIDEXPRG', 'DMDHHSIZ', 'DMDHRGND', 'DMDHRAGZ', 'DMDHREDZ', 'DMDHRMAZ', 'DMDHSEDZ', 'WTINT2YR', 'WTMEC2YR', 'SDMVSTRA', 'SDMVPSU', 'INDFMPIR']

Valores nulos por columna:
Unnamed: 0        0
SEQN              0
SDDSRVYR          0
RIDSTATR          0
RIAGENDR          0
RIDAGEYR          0
RIDAGEMN      11556
RIDRETH1          0
RIDRETH3          0
RIDEXMON       3073
RIDEXAGM       9146
DMQMILIZ       3632
DMDBORN4         19
DMDYRUSR      10058
DMDEDUC2       4139
DMDMARTZ       4141
RIDEXPRG      10430
DMDHHSIZ          0
DMDHRGND       7818
DMDHRAGZ       7809
DMDHREDZ       8187
DMDHRMAZ       7913
DMDHSEDZ       9806
WTINT2YR          0
WTMEC2YR          0
SDMVSTRA          0
SDMVPSU           0
INDFMPIR       2041
dtype: int64

Estadísticas descriptivas de variables cl

,RIDAGEYR,DMDHHSIZ,INDFMPIR
count,1.193300e+04,11933.000000,9.892000e+03
mean,3.831786e+01,3.242772,2.708174e+00
std,2.560199e+01,1.699512,1.670119e+00
min,5.397605e-79,1.000000,5.397605e-79
25%,1.300000e+01,2.000000,1.180000e+00
50%,3.700000e+01,3.000000,2.500000e+00
75%,6.200000e+01,4.000000,4.500000e+00
max,8.000000e+01,7.000000,5.000000e+00


In [44]:
#selección de columnas relevantes
features = [

    'RIAGENDR',   # Género (1=Hombre, 2=Mujer)
    'RIDAGEYR',   # Edad en años
    'RIDRETH3',   # Raza / etnia
    'DMDBORN4',   # País de nacimiento
    'DMDEDUC2',   # Nivel educativo
    'DMDMARTZ',   # Estado civil
    'DMDHHSIZ',   # Tamaño del hogar

]

target = "INDFMPIR"

# Crear copia con solo columnas necesarias
df = data[features + [target]].copy()

print("Columnas seleccionadas:", df.shape)

Columnas seleccionadas: (11933, 8)


In [45]:
#Limpieza de datos
#eliminar filas donde el target es nulo
df = df.dropna(subset=[target])

print("Filas después de eliminar nulos:", len(df))
#reemplazar valores especiales
for col in ['DMDEDUC2','DMDMARTZ','DMDBORN4']:

    df[col] = df[col].replace([7,9,77,99], np.nan)


print("Valores especiales convertidos a NaN")
#imputar valores faltantes

df['DMDEDUC2'] = df['DMDEDUC2'].fillna(
    df['DMDEDUC2'].median()
)

df['DMDMARTZ'] = df['DMDMARTZ'].fillna(
    df['DMDMARTZ'].mode()[0]
)

df['DMDBORN4'] = df['DMDBORN4'].fillna(
    df['DMDBORN4'].mode()[0]
)

print("Nulos imputados")

print(df.isnull().sum())

Filas después de eliminar nulos: 9892
Valores especiales convertidos a NaN
Nulos imputados
RIAGENDR    0
RIDAGEYR    0
RIDRETH3    0
DMDBORN4    0
DMDEDUC2    0
DMDMARTZ    0
DMDHHSIZ    0
INDFMPIR    0
dtype: int64


In [46]:
class ModeloPersonalizado(torch.nn.Module):
    def __init__(self, D_in, H, D_out):
        super(ModeloPersonalizado, self).__init__()
        self.fc1  = torch.nn.Linear(D_in, H)
        self.relu = torch.nn.ReLU()
        self.fc2  = torch.nn.Linear(H, D_out)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


In [51]:
#Checkpoint automático
def fit(model, dataloader, epochs=5, PATH="./checkpoint.pt"):
   #Entrena el modelo y guarda automáticamente el mejor checkpoint.
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = torch.nn.CrossEntropyLoss()
    best_acc  = 0

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss, train_acc = [], []
        bar = tqdm(dataloader['train'])

        for batch in bar:
            X, y = batch
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            y_hat = model(X)
            loss  = criterion(y_hat, y)
            loss.backward()
            optimizer.step()
            train_loss.append(loss.item())
            acc = (y == torch.argmax(y_hat, axis=1)).sum().item() / len(y)
            train_acc.append(acc)
            bar.set_description(f"loss {np.mean(train_loss):.5f} acc {np.mean(train_acc):.5f}")
        val_loss, val_acc = [], []
        model.eval()
        with torch.no_grad():
            bar = tqdm(dataloader['test'])
            for batch in bar:
                X, y = batch
                X, y = X.to(device), y.to(device)
                y_hat = model(X)
                loss  = criterion(y_hat, y)
                val_loss.append(loss.item())
                acc = (y == torch.argmax(y_hat, axis=1)).sum().item() / len(y)
                val_acc.append(acc)
                bar.set_description(f"val_loss {np.mean(val_loss):.5f} val_acc {np.mean(val_acc):.5f}")
# guardar modelo si es el mejor
        val_acc = np.mean(val_acc)
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), PATH)
            print(f"Best model saved at epoch {epoch} with val_acc {val_acc:.5f}")

        print(f"Epoch {epoch}/{epochs} loss {np.mean(train_loss):.5f} "
              f"val_loss {np.mean(val_loss):.5f} acc {np.mean(train_acc):.5f} val_acc {val_acc:.5f}")

    # al terminar carga el mejor modelo
    model.load_state_dict(torch.load(PATH))
#La función principal de entrenamiento. Por cada época: entrena con train, evalúa con test, y si el accuracy mejora guarda el modelo. Al final carga automáticamente el mejor.
def evaluate(model, dataloader):
    model.eval()
    model.to(device)
    bar = tqdm(dataloader['test'])
    acc = []
    with torch.no_grad():
        for X, y in bar:
            X, y  = X.to(device), y.to(device)
            y_hat = model(X)
            acc.append((y == torch.argmax(y_hat, dim=1)).sum().item() / len(y))
            bar.set_description(f"acc {np.mean(acc):.5f}")
    return np.mean(acc)

def guardar(model, PATH="./checkpoint.pt"):
    torch.save(model.state_dict(), PATH)
    print(f"Guardado en {PATH}")

def cargar(model, PATH="./checkpoint.pt"):
    model.load_state_dict(torch.load(PATH, map_location=device))
    model.eval()
    print(f"Cargado desde {PATH}")
    return model


In [52]:
class NhanesDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, ix):
        return self.X[ix], self.y[ix]

dataloader = {
    'train': torch.utils.data.DataLoader(NhanesDataset(X_train, y_train), batch_size=256, shuffle=True),
    'test':  torch.utils.data.DataLoader(NhanesDataset(X_test,  y_test),  batch_size=256, shuffle=False)
}
print(f"D_in: {X_train.shape[1]}  |  Batches train: {len(dataloader['train'])}  |  Batches test: {len(dataloader['test'])}")


D_in: 27  |  Batches train: 35  |  Batches test: 12


In [53]:
modelo = ModeloPersonalizado(D_in=X_train.shape[1], H=128, D_out=2).to(device)
print(modelo)

ModeloPersonalizado(
  (fc1): Linear(in_features=27, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=2, bias=True)
)


In [54]:
fit(modelo, dataloader, epochs=50, PATH="./delivery_model.pt")

val_loss nan val_acc 0.83541: 100%|██████████| 12/12 [00:00<00:00, 181.12it/s]


Best model saved at epoch 1 with val_acc 0.83541
Epoch 1/50 loss nan val_loss nan acc 0.83844 val_acc 0.83541


val_loss nan val_acc 0.83541: 100%|██████████| 12/12 [00:00<00:00, 190.00it/s]


Epoch 2/50 loss nan val_loss nan acc 0.83847 val_acc 0.83541


val_loss nan val_acc 0.83541: 100%|██████████| 12/12 [00:00<00:00, 136.57it/s]


...

Epoch 48/50 loss nan val_loss nan acc 0.83842 val_acc 0.83541


val_loss nan val_acc 0.83541: 100%|██████████| 12/12 [00:00<00:00, 189.88it/s]


Epoch 49/50 loss nan val_loss nan acc 0.83839 val_acc 0.83541


val_loss nan val_acc 0.83541: 100%|██████████| 12/12 [00:00<00:00, 217.31it/s]

Epoch 50/50 loss nan val_loss nan acc 0.83837 val_acc 0.83541


In [55]:
acc = evaluate(modelo, dataloader)
print(f"Accuracy final: {acc*100:.2f}%")

acc 0.83541: 100%|██████████| 12/12 [00:00<00:00, 163.65it/s]

Accuracy final: 83.54%


In [56]:
guardar(modelo, "./nhanes.pt")

Guardado en ./nhanes.pt
